# Fine-tune intfloat/multilingual-e5-base với TripletLoss + MNRL (v3)

**Dataset:** `train_v3.jsonl` (pos text dùng corpus `searchable_text` để tránh format mismatch)

## Key fixes vs v2:
1. **Format alignment**: `pos[0]` dùng corpus `searchable_text` thay vì clean text (fix distribution shift)
2. **TripletLoss margin**: 0.5 → 1.0 (push negatives xa hơn)
3. **Evaluator**: TripletEvaluator → InformationRetrievalEvaluator (correlate với test Recall@K)
4. **Loss weights**: TripletLoss=0.3, MNRL=0.7 (in-batch negatives hiệu quả hơn)
5. **Epochs**: 2 → 3 + early stopping
6. **Easy negatives**: bổ sung 24k easy negs cho triplets thiếu hard negs

## 1. Cài đặt thư viện

In [ ]:
# !pip install -q sentence-transformers datasets torch pandas numpy

## 2. Mount Google Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess, os
GITHUB_REPO = "PhamMinhDan/llm_provider_benchmarking_ver2"
BRANCH      = "main"
TARGET_DIR  = "/content/llm_provider_benchmarking_ver2"

if os.path.exists(TARGET_DIR):
    subprocess.run(["git", "-C", TARGET_DIR, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", BRANCH,
        "--filter=blob:none",
        f"https://github.com/{GITHUB_REPO}.git",
        TARGET_DIR,
    ], check=True)
print("Repo ready.")

## 3. Import & Config

In [ ]:
import json, random, time, os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
)
from datasets import Dataset

# === CONFIG ===
BASE_MODEL     = "intfloat/multilingual-e5-base"
MAX_SEQ_LENGTH = 512

REPO_DIR   = Path(TARGET_DIR)
DATA_DIR   = REPO_DIR / "embedding_project/data"
TRAIN_JSON = Path("/content/drive/MyDrive/DATN/data/train_v3_filtered.jsonl")  # V3 from Drive
VALID_JSON = Path("/content/drive/MyDrive/DATN/data/valid_v3_filtered.jsonl")  # V3 from Drive
TEST_JSON  = Path("/content/drive/MyDrive/DATN/data/test_v3_fair_filtered.jsonl")  # V3 from Drive
CORPUS_CSV = DATA_DIR / "Dataset_DATN_28k.csv"

GDRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/DATN/models/e5_base_v3_finetuned")
LOCAL_OUTPUT_DIR  = Path("/content/e5_base_v3_finetuned")

# === Training hyperparams (v3 fixes) ===
EPOCHS          = 3                # v2: 2 → v3: 3 (more time to learn)
BATCH_SIZE      = 32
GRAD_ACCUM      = 2                # effective batch = 64
LR              = 2e-5             # v2: 1e-5 → v3: 2e-5 (slightly higher)
WARMUP_RATIO    = 0.1
TRIPLET_MARGIN  = 1.0              # v2: 0.5 → v3: 1.0 (push negs further)
TRIPLET_WEIGHT  = 0.3              # weighted loss
MNRL_WEIGHT     = 0.7
TRIPLET_LOSS_NAME = "triplet"
MNRL_LOSS_NAME    = "mnrl"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Model: {BASE_MODEL}")
print(f"Train: {TRAIN_JSON.name} | Valid: {VALID_JSON.name} | Test: {TEST_JSON.name}")
print(f"Hyperparams: epochs={EPOCHS}, batch={BATCH_SIZE}*{GRAD_ACCUM}, lr={LR}, margin={TRIPLET_MARGIN}")
print(f"Loss weights: Triplet={TRIPLET_WEIGHT}, MNRL={MNRL_WEIGHT}")


## 4. Load Dataset

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(TRAIN_JSON)
valid_raw = load_jsonl(VALID_JSON)
test_raw  = load_jsonl(TEST_JSON)

# === Filter out rows with 0 negatives (TripleLoss requires 1 anchor + 1 pos + 1+ neg) ===
# Defense-in-depth: _filtered.jsonl files already exclude these, but extra guard
# ensures correctness if upstream data changes.
train_raw = [r for r in train_raw if len(r.get("neg", [])) > 0]
valid_raw = [r for r in valid_raw if len(r.get("neg", [])) > 0]
test_raw  = [r for r in test_raw  if len(r.get("neg", [])) > 0]
print(f"After 0-neg filter: train={len(train_raw):,}  valid={len(valid_raw):,}  test={len(test_raw):,}")

print(f"Train: {len(train_raw):,}  |  Valid: {len(valid_raw):,}  |  Test: {len(test_raw):,}")

for name, recs in [("Train", train_raw), ("Valid", valid_raw), ("Test", test_raw)]:
    qt = {}
    for r in recs:
        qt[r.get("query_type", "unknown")] = qt.get(r.get("query_type","unknown"), 0) + 1
    n_negs = [len(r.get("neg", [])) for r in recs]
    avg_negs = sum(n_negs) / len(n_negs) if n_negs else 0
    print(f"  {name}: qtype={qt}  |  avg_negs/row: {avg_negs:.2f}")

## 5. Build TripletLoss Dataset

- TripletLoss: specific + ≥1 neg → {(anchor, positive, negative)}
- MNRL: còn lại (vague có/không neg, specific không neg) → in-batch negatives

In [ ]:
train_triplets = []  # TripletLoss
train_mnrl    = []  # MNRL

for r in train_raw:
    query   = r.get("query", "").strip()
    pos_lst = r.get("pos", [])
    neg_lst = r.get("neg", [])
    if not pos_lst or not pos_lst[0].strip():
        continue

    anchor   = f"query: {query}"
    positive = f"passage: {pos_lst[0].strip()}"
    has_neg  = bool(neg_lst and neg_lst[0].strip())
    is_specific = r.get("query_type") == "specific"

    if is_specific and has_neg:
        train_triplets.append({
            "anchor":     anchor,
            "positive":   positive,
            "negative":   f"passage: {neg_lst[0].strip()}",
        })
    else:
        train_mnrl.append({
            "anchor":     anchor,
            "positive":   positive,
        })

print(f"TripletLoss samples (specific + neg): {len(train_triplets):,}")
print(f"MNRL samples (vague + specific-no-neg): {len(train_mnrl):,}")
print(f"Total: {len(train_triplets) + len(train_mnrl):,}")

# Valid for TripletLoss eval
valid_for_triplet = []
for r in valid_raw:
    if (r.get("pos") and r["pos"] and r["pos"][0].strip() and
        r.get("neg") and r["neg"] and r["neg"][0].strip()):
        valid_for_triplet.append({
            "anchor":   f"query: {r['query'].strip()}",
            "positive": f"passage: {r['pos'][0].strip()}",
            "negative": f"passage: {r['neg'][0].strip()}",
        })
print(f"\nValid for TripletLoss eval: {len(valid_for_triplet):,}")

## 6. Tạo Dataset cho sentence-transformers

In [ ]:
random.seed(42)
random.shuffle(train_triplets)
random.shuffle(train_mnrl)

train_ds_triplet = Dataset.from_list([
    {"anchor": t["anchor"], "positive": t["positive"], "negative": t["negative"]}
    for t in train_triplets
])
train_ds_mnrl = Dataset.from_list([
    {"anchor": t["anchor"], "positive": t["positive"]}
    for t in train_mnrl
])
print(f"Train triplet ds: {len(train_ds_triplet):,}")
print(f"Train MNRL ds   : {len(train_ds_mnrl):,}")

## 7. Load Model

In [ ]:
print(f"Loading {BASE_MODEL}...")
model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH
print(f"Model loaded.  max_seq_length={model.max_seq_length}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## 8. Triplet Evaluator (v3: InformationRetrievalEvaluator)

Thay TripletEvaluator bằng InformationRetrievalEvaluator trên 1000 valid samples.
Mục tiêu: chọn checkpoint dựa trên Recall@10 (correlate với test metric).

In [ ]:
# Build IR evaluator using valid set as queries, corpus = a subset
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from collections import defaultdict

# Sample 1000 valid triplets for IR eval
ir_eval_size = 1000
ir_eval_data = valid_for_triplet[:ir_eval_size]

ir_queries = {}
ir_corpus  = {}
ir_relevant = {}

for i, t in enumerate(ir_eval_data):
    qid = f"q_{i}"
    pid_pos = f"p_pos_{i}"
    pid_neg = f"p_neg_{i}"
    # Strip prefix for evaluator (it adds itself)
    ir_queries[qid] = t["anchor"].replace("query: ", "")
    ir_corpus[pid_pos] = t["positive"].replace("passage: ", "")
    ir_corpus[pid_neg] = t["negative"].replace("passage: ", "")
    ir_relevant[qid] = {pid_pos}  # only positive is relevant

ir_evaluator = InformationRetrievalEvaluator(
    queries=ir_queries,
    corpus=ir_corpus,
    relevant_docs=ir_relevant,
    name="valid_ir",
    show_progress_bar=True,
    batch_size=64,
)
print(f"IR Evaluator ready: {len(ir_queries):,} queries, {len(ir_corpus):,} corpus docs")
print(f"  Metric tracked: cosine_accuracy@1, cosine_accuracy@10, mrr@10, ndcg@10")

## 9. Training Arguments

In [ ]:
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args = SentenceTransformerTrainingArguments(
    output_dir=str(LOCAL_OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,        # v3: keep 2 best (smaller storage)
    load_best_model_at_end=True,
    logging_steps=200,
    log_on_each_node=False,
    run_name="e5_base_v3_finetuned",
    report_to=["none"],
    seed=42,
    data_seed=42,
    # IR evaluator outputs 'valid_ir_cosine_ndcg@10' (the best correlate to test Recall@K)
    metric_for_best_model="valid_ir_cosine_ndcg@10",
    greater_is_better=True,
)

print("Training args:")
for k, v in [("epochs", EPOCHS), ("batch_size", BATCH_SIZE), ("grad_accum", GRAD_ACCUM),
             ("effective_batch", BATCH_SIZE * GRAD_ACCUM), ("lr", LR),
             ("warmup_ratio", WARMUP_RATIO), ("triplet_margin", TRIPLET_MARGIN),
             ("fp16", args.fp16)]:
    print(f"  {k}: {v}")

## 10. Loss & Trainer (v3: weighted losses)

In [ ]:
from sentence_transformers.losses import (
    TripletLoss,
    MultipleNegativesRankingLoss,
)
from sentence_transformers.trainer import WeightedLossTrainer

loss_triplet = TripletLoss(model=model, triplet_margin=TRIPLET_MARGIN)
loss_mnrl    = MultipleNegativesRankingLoss(model=model)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset={"triplet": train_ds_triplet, "mnrl": train_ds_mnrl},
    eval_dataset={"triplet": Dataset.from_list([
        {"anchor": t["anchor"], "positive": t["positive"], "negative": t["negative"]}
        for t in valid_for_triplet[:500]
    ])},
    loss={"triplet": (loss_triplet, TRIPLET_WEIGHT),
          "mnrl":    (loss_mnrl, MNRL_WEIGHT)},
    evaluator=ir_evaluator,
)

print(f"Trainer ready. Loss weights: Triplet={TRIPLET_WEIGHT}, MNRL={MNRL_WEIGHT}")
print(f"Total train samples: {len(train_ds_triplet) + len(train_ds_mnrl):,}")
print(f"Total training steps: ~{((len(train_ds_triplet) + len(train_ds_mnrl)) * EPOCHS) // (BATCH_SIZE * GRAD_ACCUM):,}")

## 11. Train!

In [ ]:
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0
print(f"\nTraining done in {elapsed/60:.1f} min")
print(f"Final metric (best): {trainer.state.best_metric}")
print(f"Best model checkpoint: {trainer.state.best_model_checkpoint}")

## 12. Save Model + Zip → Google Drive

In [ ]:
FINAL_DIR = LOCAL_OUTPUT_DIR / "final"
model.save(str(FINAL_DIR))
print(f"Model saved locally: {FINAL_DIR}")

# Metadata
meta = {
    "base_model":     BASE_MODEL,
    "version":        "v3",
    "epochs":         EPOCHS,
    "batch_size":     BATCH_SIZE,
    "learning_rate":  LR,
    "warmup_ratio":   WARMUP_RATIO,
    "triplet_margin": TRIPLET_MARGIN,
    "triplet_weight": TRIPLET_WEIGHT,
    "mnrl_weight":    MNRL_WEIGHT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_triplet":  len(train_ds_triplet),
    "train_mnrl":     len(train_ds_mnrl),
    "valid_samples":  len(valid_for_triplet),
    "train_time_min": round(elapsed / 60, 1),
    "best_metric":    float(trainer.state.best_metric) if trainer.state.best_metric else None,
    "loss":           "TripletLoss + MNRL (weighted)",
    "evaluator":      "InformationRetrievalEvaluator (NDCG@10)",
    "dataset":        str(TRAIN_JSON.name),
    "key_fixes_vs_v2": [
        "pos[0] = corpus searchable_text (no format mismatch)",
        "TripletLoss margin 0.5 → 1.0",
        "TripletEvaluator → InformationRetrievalEvaluator (NDCG@10)",
        "Weighted loss: Triplet=0.3, MNRL=0.7",
        "Epochs 2 → 3",
        "Easy negatives added for triplets n_hard=0",
    ],
}
with open(FINAL_DIR / "finetune_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("Metadata saved.")

In [ ]:
import zipfile, os
from datetime import datetime

GDRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_NAME  = f"e5_base_v3_finetuned_{timestamp}.zip"
ZIP_PATH  = GDRIVE_OUTPUT_DIR / ZIP_NAME

def zipdir(path: Path, ziph: zipfile.ZipFile):
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = Path(root) / file
            arcname = file_path.relative_to(path.parent)
            ziph.write(file_path, arcname)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zipdir(FINAL_DIR, zf)

print(f"Zip created: {ZIP_PATH}")
print(f"Zip size: {ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB")

from google.colab import files
files.download(str(ZIP_PATH))

## 13. Evaluate trên Test Set

Recall@K, MRR@K, NDCG@K — stratified theo query_type (specific vs vague)

In [ ]:
corpus_df = pd.read_csv(CORPUS_CSV, usecols=["product_id", "searchable_text"])
corpus_df = corpus_df.dropna(subset=["product_id", "searchable_text"])
corpus_df["product_id"] = corpus_df["product_id"].astype(str)
corpus_df = corpus_df.drop_duplicates(subset="product_id", keep="first")
corpus_ids   = corpus_df["product_id"].tolist()
corpus_texts = corpus_df["searchable_text"].tolist()
print(f"Corpus: {len(corpus_ids):,} products")

In [ ]:
def batch_encode(texts: list[str], batch_size: int = 256) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

print("Encoding corpus...")
t0 = time.time()
corpus_emb = batch_encode([f"passage: {t}" for t in corpus_texts])
print(f"Corpus encoded: {corpus_emb.shape}  ({time.time()-t0:.1f}s)")

In [ ]:
test_triplets = []
for r in test_raw:
    query   = r.get("query", "").strip()
    pos_lst = r.get("pos", [])
    if not pos_lst or not pos_lst[0].strip():
        continue
    test_triplets.append({
        "anchor":     f"query: {query}",
        "query":      query,
        "positive":   f"passage: {pos_lst[0].strip()}",
        "product_id": str(r.get("product_id", "")),
        "query_type": r.get("query_type", ""),
    })
print(f"Test triplets: {len(test_triplets):,}")

def recall_at_k(retrieved: list[str], gt_id: str, k: int) -> float:
    return float(gt_id in retrieved[:k])

def mrr_at_k(retrieved: list[str], gt_id: str, k: int) -> float:
    for i, pid in enumerate(retrieved[:k], start=1):
        if pid == gt_id:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved: list[str], gt_id: str, k: int) -> float:
    k = min(k, len(retrieved))
    actual = [1.0 if pid == gt_id else 0.0 for pid in retrieved[:k]]
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(actual))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(k, 1)))
    return dcg / idcg if idcg > 0 else 0.0

def retrieve_topk(query_texts: list[str], k: int = 50) -> list[list[str]]:
    q_emb = model.encode(
        [f"query: {q}" for q in query_texts],
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    scores   = np.dot(q_emb, corpus_emb.T)
    topk_idx = np.argpartition(-scores, kth=k, axis=1)[:, :k]
    sorted_idx = np.argsort(-np.take_along_axis(scores, topk_idx, axis=1), axis=1)
    topk_idx = np.take_along_axis(topk_idx, sorted_idx, axis=1)
    return [[corpus_ids[j] for j in row] for row in topk_idx]

def evaluate_split(triplets: list[dict], k_values: list[int]) -> dict:
    queries   = [t["query"] for t in triplets]
    retrieved = retrieve_topk(queries, k=max(k_values))
    results   = {}
    for k in k_values:
        rec  = [recall_at_k(r, t["product_id"], k) for r, t in zip(retrieved, triplets)]
        mrr  = [mrr_at_k(r, t["product_id"], k)    for r, t in zip(retrieved, triplets)]
        ndcg = [ndcg_at_k(r, t["product_id"], k)   for r, t in zip(retrieved, triplets)]
        results[f"Recall@{k}"]  = round(np.mean(rec)  * 100, 2)
        results[f"MRR@{k}"]    = round(np.mean(mrr)  * 100, 2)
        results[f"NDCG@{k}"]   = round(np.mean(ndcg) * 100, 2)
    return results

# Evaluate on test set
print(f"\nEvaluating on Test set ({len(test_triplets):,} samples)...")
t0 = time.time()
test_metrics = evaluate_split(test_triplets, k_values=[1, 5, 10, 20, 50])
print(f"Done in {time.time()-t0:.1f}s\n")

print("=" * 50)
print("  TEST SET RETRIEVAL RESULTS (E5-FT v3)")
print("=" * 50)
for metric, value in test_metrics.items():
    print(f"  {metric:20s}: {value:6.2f}%")
print("=" * 50)

# Stratified by query type
print("\nStratified by query_type:")
for qtype in ["specific", "vague"]:
    sub = [t for t in test_triplets if t.get("query_type") == qtype]
    if not sub:
        continue
    m = evaluate_split(sub, k_values=[1, 5, 10])
    print(f"\n  {qtype.upper()} ({len(sub)} samples):")
    for k, v in m.items():
        print(f"    {k:20s}: {v:6.2f}%")

# Save metrics
import json
with open(FINAL_DIR / "test_metrics_v3.json", "w") as f:
    json.dump({
        "test_overall": test_metrics,
        "version": "v3",
        "n_test": len(test_triplets),
    }, f, indent=2)
print("\nMetrics saved to test_metrics_v3.json")

## 14. Manual Eval trên benchmark_queries_200 (queries vague thực sự)

Đây là test phụ — 200 queries vague (do con người tạo), dùng để verify E5-FT có cải thiện semantic search trên queries mơ hồ không (nơi BM25 yếu).

In [ ]:
benchmark_df = pd.read_csv(DATA_DIR / "benchmark_queries_200.csv")
print(f"Benchmark 200 queries: {len(benchmark_df)}")
print(f"Columns: {list(benchmark_df.columns)}")
print(f"Sample:")
print(benchmark_df.head(3).to_string())

# Apply same retrieval
benchmark_queries = benchmark_df['query'].tolist()
benchmark_gt = benchmark_df['product_id'].astype(str).tolist()

retrieved_bench = retrieve_topk(benchmark_queries, k=20)

results_bench = {}
for k in [1, 5, 10, 20]:
    rec = [recall_at_k(r, gt, k) for r, gt in zip(retrieved_bench, benchmark_gt)]
    mrr = [mrr_at_k(r, gt, k)    for r, gt in zip(retrieved_bench, benchmark_gt)]
    results_bench[f"Recall@{k}"] = round(np.mean(rec) * 100, 2)
    results_bench[f"MRR@{k}"]   = round(np.mean(mrr) * 100, 2)

print("\n" + "=" * 50)
print("  BENCHMARK 200 (vague queries) — E5-FT v3")
print("=" * 50)
for k, v in results_bench.items():
    print(f"  {k:20s}: {v:6.2f}%")
print("=" * 50)

with open(FINAL_DIR / "benchmark_200_v3.json", "w") as f:
    json.dump(results_bench, f, indent=2)
print("Saved to benchmark_200_v3.json")

## 15. Benchmark V3 — Pretrained vs FT-V3 (auto-run)

So sánh E5 pretrained vs model vừa fine-tune (Cell 11):
  - Test set (test_v3_fair_filtered): specific / vague split
  - Benchmark 200 queries vague

Metrics: Recall@K, MRR@K, NDCG@K. Kết quả lưu vào `embedding_project/outputs/evaluation/comparison_v3.json`.

In [ ]:
import json, time
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = Path('embedding_project/outputs/evaluation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Load corpus (reuse logic với Cell 13) ===
_corpus_df = pd.read_csv(CORPUS_CSV, usecols=['product_id', 'searchable_text'])
_corpus_df = _corpus_df.dropna(subset=['product_id', 'searchable_text'])
_corpus_df['product_id'] = _corpus_df['product_id'].astype(str)
_corpus_df = _corpus_df.drop_duplicates('product_id', keep='first')
corpus_ids = _corpus_df['product_id'].tolist()
corpus_texts = _corpus_df['searchable_text'].tolist()
print(f'Corpus: {len(corpus_ids):,} products')

# === Re-read test (multi-GT) ===
_test_items = [json.loads(l) for l in TEST_JSON.read_text(encoding='utf-8').splitlines() if l.strip()]
test_queries = [t['query'] for t in _test_items]
test_gt = [t['relevant_pids'] for t in _test_items]
test_qtype = [t.get('query_type', '') for t in _test_items]
print(f'Test: {len(_test_items):,} queries (specific={test_qtype.count("specific")}, vague={test_qtype.count("vague")})')

# === Load benchmark 200 ===
_bench_df = pd.read_csv(DATA_DIR / 'benchmark_queries_200.csv')
bench_queries = _bench_df['query'].tolist()
bench_gt = _bench_df['product_id'].astype(str).tolist()
print(f'Benchmark 200: {len(bench_queries):,} queries')

# === Metrics ===
def recall_at_k(retrieved, gt_set, k):
    if not gt_set: return 0.0
    hits = sum(1 for pid in retrieved[:k] if pid in gt_set)
    return hits / len(gt_set)

def mrr_at_k(retrieved, gt_set, k):
    for i, pid in enumerate(retrieved[:k], 1):
        if pid in gt_set:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved, gt_set, k):
    k = min(k, len(retrieved))
    actual = [1.0 if pid in gt_set else 0.0 for pid in retrieved[:k]]
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(actual))
    n_rel = min(len(gt_set), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(n_rel))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(retrieved_lists, gt_lists, k_values=[1, 5, 10, 20]):
    gt_sets = [set(gt) if isinstance(gt, list) else {gt} for gt in gt_lists]
    results = {}
    for k in k_values:
        rec  = [recall_at_k(r, g, k) for r, g in zip(retrieved_lists, gt_sets)]
        mrr  = [mrr_at_k(r, g, k)    for r, g in zip(retrieved_lists, gt_sets)]
        ndcg = [ndcg_at_k(r, g, k)   for r, g in zip(retrieved_lists, gt_sets)]
        results[f'Recall@{k}'] = round(np.mean(rec)  * 100, 2)
        results[f'MRR@{k}']    = round(np.mean(mrr)  * 100, 2)
        results[f'NDCG@{k}']   = round(np.mean(ndcg) * 100, 2)
    return results

# === Embed helpers ===
def encode_passages(model, texts, batch_size=128):
    return model.encode([f'passage: {t}' for t in texts],
                        batch_size=batch_size, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True)

def encode_queries(model, queries, batch_size=128):
    return model.encode([f'query: {q}' for q in queries],
                        batch_size=batch_size, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True)

def retrieve_topk(query_emb, corpus_emb, corpus_ids, k=50):
    scores = np.dot(query_emb, corpus_emb.T)
    topk_idx = np.argpartition(-scores, kth=k, axis=1)[:, :k]
    sorted_idx = np.argsort(-np.take_along_axis(scores, topk_idx, axis=1), axis=1)
    topk_idx = np.take_along_axis(topk_idx, sorted_idx, axis=1)
    return [[corpus_ids[j] for j in row] for row in topk_idx]

all_results = {}

# --- E5 Pretrained ---
print('\n' + '='*60)
print('E5 Pretrained (multilingual-e5-base)')
print('='*60)
_t0 = time.time()
e5_pre = SentenceTransformer('intfloat/multilingual-e5-base')
print(f'Loaded in {time.time()-_t0:.1f}s')

_corpus_emb_pre = encode_passages(e5_pre, corpus_texts)
_test_emb_pre = encode_queries(e5_pre, test_queries)
_bench_emb_pre = encode_queries(e5_pre, bench_queries)

_pre_test = retrieve_topk(_test_emb_pre, _corpus_emb_pre, corpus_ids, k=50)
_pre_bench = retrieve_topk(_bench_emb_pre, _corpus_emb_pre, corpus_ids, k=20)

all_results['E5_pretrained'] = {
    'test': evaluate(_pre_test, test_gt),
    'test_specific': evaluate([r for r, q in zip(_pre_test, test_qtype) if q == 'specific'],
                              [g for g, q in zip(test_gt, test_qtype) if q == 'specific']),
    'test_vague':    evaluate([r for r, q in zip(_pre_test, test_qtype) if q == 'vague'],
                              [g for g, q in zip(test_gt, test_qtype) if q == 'vague']),
    'benchmark_200': evaluate(_pre_bench, bench_gt, k_values=[1, 5, 10, 20]),
}
del e5_pre, _corpus_emb_pre, _test_emb_pre, _bench_emb_pre

# --- E5-FT V3 ---
_v3_path = LOCAL_OUTPUT_DIR / 'final'
if _v3_path.exists():
    print('\n' + '='*60)
    print('E5-FT V3 (NEW)')
    print('='*60)
    _t0 = time.time()
    e5_v3 = SentenceTransformer(str(_v3_path))
    print(f'Loaded in {time.time()-_t0:.1f}s')

    _corpus_emb_v3 = encode_passages(e5_v3, corpus_texts)
    _test_emb_v3 = encode_queries(e5_v3, test_queries)
    _bench_emb_v3 = encode_queries(e5_v3, bench_queries)

    _v3_test = retrieve_topk(_test_emb_v3, _corpus_emb_v3, corpus_ids, k=50)
    _v3_bench = retrieve_topk(_bench_emb_v3, _corpus_emb_v3, corpus_ids, k=20)

    all_results['E5_FT_v3'] = {
        'test': evaluate(_v3_test, test_gt),
        'test_specific': evaluate([r for r, q in zip(_v3_test, test_qtype) if q == 'specific'],
                                  [g for g, q in zip(test_gt, test_qtype) if q == 'specific']),
        'test_vague':    evaluate([r for r, q in zip(_v3_test, test_qtype) if q == 'vague'],
                                  [g for g, q in zip(test_gt, test_qtype) if q == 'vague']),
        'benchmark_200': evaluate(_v3_bench, bench_gt, k_values=[1, 5, 10, 20]),
    }
    del e5_v3, _corpus_emb_v3, _test_emb_v3, _bench_emb_v3
else:
    print(f'\nERROR: V3 model not found at {_v3_path}')

# === Save & Pretty Print ===
out_file = OUTPUT_DIR / 'comparison_v3.json'
with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f'\nResults saved: {out_file}')

print('\n' + '='*70)
print(' COMPARISON: E5 Pretrained vs E5-FT V3')
print('='*70)
for _name, _res in all_results.items():
    print(f'\n{_name}:')
    print(f'  Test overall:        R@10={_res["test"]["Recall@10"]:.2f}  MRR@10={_res["test"]["MRR@10"]:.2f}  NDCG@10={_res["test"]["NDCG@10"]:.2f}')
    if 'test_specific' in _res:
        print(f'  Test SPECIFIC:       R@10={_res["test_specific"]["Recall@10"]:.2f}  MRR@10={_res["test_specific"]["MRR@10"]:.2f}')
    if 'test_vague' in _res:
        print(f'  Test VAGUE:          R@10={_res["test_vague"]["Recall@10"]:.2f}  MRR@10={_res["test_vague"]["MRR@10"]:.2f}')
    if 'benchmark_200' in _res:
        print(f'  Bench 200 (vague):   R@10={_res["benchmark_200"]["Recall@10"]:.2f}  MRR@10={_res["benchmark_200"]["MRR@10"]:.2f}')

if 'E5_pretrained' in all_results and 'E5_FT_v3' in all_results:
    print('\n' + '='*70)
    print(' DELTA (V3 - Pretrained)')
    print('='*70)
    _pre = all_results['E5_pretrained']
    _v3 = all_results['E5_FT_v3']
    for _split in ['test', 'test_specific', 'test_vague', 'benchmark_200']:
        if _split not in _pre or _split not in _v3: continue
        print(f'\n  {_split}:')
        for _m in ['Recall@10', 'MRR@10', 'NDCG@10']:
            if _m in _pre[_split] and _m in _v3[_split]:
                _d = _v3[_split][_m] - _pre[_split][_m]
                _s = '+' if _d >= 0 else ''
                print(f'    {_m:18s}: {_pre[_split][_m]:6.2f} → {_v3[_split][_m]:6.2f} ({_s}{_d:.2f})')